# Playground Series S6E4 - Predicting Irrigation Need EDA

This notebook is designed to run inside Kaggle only. Attach the competition data from `playground-series-s6e4`, then run all cells.

Goals:
- verify the files and schema
- understand target balance
- inspect missingness, feature distributions, and train/test differences
- identify a few early modeling risks and promising features

## 1. Imports and Display Settings

In [ ]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:,.4f}'.format)

sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42

## 2. Locate Competition Files

In [ ]:
INPUT_ROOT = Path('/kaggle/input')

if not INPUT_ROOT.exists():
    raise RuntimeError('This notebook is intended to run on Kaggle, where /kaggle/input is available.')

def find_file(filename: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Could not find {filename} under {INPUT_ROOT}. Attach the competition data to this notebook.')
    if len(matches) > 1:
        print(f'Found multiple {filename} files. Using: {matches[0]}')
        for match in matches:
            print(' -', match)
    return matches[0]

train_path = find_file('train.csv')
test_path = find_file('test.csv')
sample_submission_path = find_file('sample_submission.csv')

print('train:', train_path)
print('test:', test_path)
print('sample submission:', sample_submission_path)

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

target_col = sample_submission.columns[-1]
id_col = sample_submission.columns[0]

print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')
print(f'sample submission shape: {sample_submission.shape}')
print(f'id column: {id_col}')
print(f'target column: {target_col}')

## 3. First Look

In [ ]:
display(train.head())
display(test.head())
display(sample_submission.head())

In [ ]:
overview = pd.DataFrame({
    'train_dtype': train.dtypes.astype(str),
    'test_dtype': test.dtypes.astype(str).reindex(train.columns),
    'train_missing': train.isna().sum(),
    'train_missing_pct': train.isna().mean() * 100,
    'train_unique': train.nunique(dropna=False),
})
display(overview.sort_values(['train_missing_pct', 'train_unique'], ascending=[False, True]))

In [ ]:
feature_cols = [c for c in train.columns if c not in {id_col, target_col}]
numeric_cols = train[feature_cols].select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print(f'Number of features: {len(feature_cols)}')
print(f'Numeric features: {len(numeric_cols)}')
print(f'Categorical / object features: {len(categorical_cols)}')
print(f'Duplicate rows in train: {train.duplicated().sum()}')
print(f'Duplicate ids in train: {train[id_col].duplicated().sum() if id_col in train else "id not found"}')
print(f'Duplicate ids in test: {test[id_col].duplicated().sum() if id_col in test else "id not found"}')

## 4. Target Distribution

In [ ]:
target_counts = train[target_col].value_counts(dropna=False)
target_pct = train[target_col].value_counts(normalize=True, dropna=False).mul(100)
target_summary = pd.concat([target_counts.rename('count'), target_pct.rename('pct')], axis=1)
display(target_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=train, x=target_col, order=target_counts.index, ax=ax)
ax.set_title('Target Distribution')
ax.set_xlabel(target_col)
ax.set_ylabel('Rows')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 5. Missingness

In [ ]:
missing = pd.DataFrame({
    'train_missing_pct': train[feature_cols].isna().mean().mul(100),
    'test_missing_pct': test[[c for c in feature_cols if c in test.columns]].isna().mean().mul(100),
}).fillna(0)
missing['abs_diff_pct'] = (missing['train_missing_pct'] - missing['test_missing_pct']).abs()

display(missing.sort_values(['train_missing_pct', 'abs_diff_pct'], ascending=False).head(30))

plot_missing = missing[(missing['train_missing_pct'] > 0) | (missing['test_missing_pct'] > 0)].sort_values('train_missing_pct', ascending=False)
if plot_missing.empty:
    print('No missing values found in train/test features.')
else:
    ax = plot_missing[['train_missing_pct', 'test_missing_pct']].head(40).plot(kind='bar', figsize=(12, 5))
    ax.set_title('Missing Values by Feature')
    ax.set_ylabel('Missing %')
    ax.set_xlabel('Feature')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Numeric Feature Summary

In [ ]:
if numeric_cols:
    display(train[numeric_cols].describe().T)
else:
    print('No numeric features detected.')

In [ ]:
def plot_numeric_histograms(df, cols, max_cols=16):
    cols = cols[:max_cols]
    if not cols:
        print('No numeric columns to plot.')
        return
    ncols = 4
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.2 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, cols):
        sns.histplot(df[col], bins=40, kde=False, ax=ax)
        ax.set_title(col)
        ax.set_xlabel('')
    for ax in axes[len(cols):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

plot_numeric_histograms(train, numeric_cols, max_cols=24)

In [ ]:
if numeric_cols:
    corr = train[numeric_cols].corr(numeric_only=True)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    plt.figure(figsize=(min(18, 1 + 0.45 * len(numeric_cols)), min(14, 1 + 0.45 * len(numeric_cols))))
    sns.heatmap(corr, mask=mask, cmap='vlag', center=0, linewidths=0.3, cbar_kws={'shrink': 0.8})
    plt.title('Numeric Feature Correlations')
    plt.tight_layout()
    plt.show()
else:
    print('No numeric features detected.')

## 7. Feature Behavior by Target

In [ ]:
if numeric_cols:
    top_numeric = numeric_cols[:12]
    ncols = 3
    nrows = int(np.ceil(len(top_numeric) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, top_numeric):
        sns.boxplot(data=train, x=target_col, y=col, ax=ax, showfliers=False)
        ax.set_title(col)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=20)
    for ax in axes[len(top_numeric):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No numeric features detected.')

In [ ]:
if categorical_cols:
    for col in categorical_cols[:8]:
        ct = pd.crosstab(train[col], train[target_col], normalize='index').sort_index()
        display(ct.head(30))
        ax = ct.head(30).plot(kind='bar', stacked=True, figsize=(12, 4), colormap='tab20')
        ax.set_title(f'Target Mix by {col}')
        ax.set_ylabel('Share within category')
        ax.set_xlabel(col)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('No categorical features detected.')

## 8. Train/Test Distribution Checks

In [ ]:
common_numeric = [c for c in numeric_cols if c in test.columns]

drift_rows = []
for col in common_numeric:
    train_mean = train[col].mean()
    test_mean = test[col].mean()
    train_std = train[col].std()
    test_std = test[col].std()
    pooled_std = pd.concat([train[col], test[col]], axis=0).std()
    standardized_mean_diff = np.nan if pooled_std == 0 else abs(train_mean - test_mean) / pooled_std
    drift_rows.append({
        'feature': col,
        'train_mean': train_mean,
        'test_mean': test_mean,
        'train_std': train_std,
        'test_std': test_std,
        'standardized_mean_diff': standardized_mean_diff,
    })

drift = pd.DataFrame(drift_rows).sort_values('standardized_mean_diff', ascending=False)
display(drift.head(30))

In [ ]:
if common_numeric:
    plot_cols = drift['feature'].head(12).tolist()
    ncols = 3
    nrows = int(np.ceil(len(plot_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.6 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, plot_cols):
        sns.kdeplot(train[col], label='train', ax=ax, fill=False)
        sns.kdeplot(test[col], label='test', ax=ax, fill=False)
        ax.set_title(col)
        ax.set_xlabel('')
        ax.legend()
    for ax in axes[len(plot_cols):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No common numeric features detected.')

## 9. Quick Univariate Signal Ranking

In [ ]:
try:
    from sklearn.feature_selection import mutual_info_classif
    from sklearn.preprocessing import OrdinalEncoder

    X = train[feature_cols].copy()
    y = train[target_col]

    cat_cols_for_mi = X.select_dtypes(exclude=np.number).columns.tolist()
    num_cols_for_mi = X.select_dtypes(include=np.number).columns.tolist()

    for col in num_cols_for_mi:
        X[col] = X[col].fillna(X[col].median())

    if cat_cols_for_mi:
        X[cat_cols_for_mi] = X[cat_cols_for_mi].astype('string').fillna('__missing__')
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[cat_cols_for_mi] = encoder.fit_transform(X[cat_cols_for_mi])

    discrete_mask = [col in cat_cols_for_mi for col in X.columns]
    mi = mutual_info_classif(X, y, discrete_features=discrete_mask, random_state=RANDOM_STATE)
    mi_df = pd.DataFrame({'feature': X.columns, 'mutual_info': mi}).sort_values('mutual_info', ascending=False)
    display(mi_df.head(30))

    plt.figure(figsize=(10, 6))
    sns.barplot(data=mi_df.head(25), x='mutual_info', y='feature')
    plt.title('Top Features by Mutual Information')
    plt.xlabel('Mutual information')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print('Skipped mutual information ranking:', repr(exc))

## 10. EDA Notes

Use this section while running on Kaggle to capture observations:

- Target balance:
- Missingness concerns:
- Train/test drift concerns:
- Strong-looking features:
- Feature engineering ideas:
- Validation strategy thoughts: